## Demostration of the original and optimized BPE (training & inference)
- To run this notebook, you need to create a `data/` folder to store the txt data for BPE to train on (and tokenize).
    - The TinyStories dataset is not bad to try on. You can download it via [this link](https://huggingface.co/datasets/roneneldan/TinyStories)
- Also, you need to create an `outputs/` folder to store the trained BPE tokenizer files, and support the BPE inference afterwards.

In [ ]:
import numpy as np

with open("data/TinyStoriesV2-GPT4-valid.txt", "r") as f:
    lines = f.readlines()

In [2]:
len(lines)

157832

In [6]:
for i in lines[:5]:
    print(i)
    print("="*80)

u don't have to be scared of the loud dog, I'll protect you". The mole felt so safe with the little girl. She was very kind and the mole soon came to trust her. He leaned against her and she kept him safe. The mole had found his best friend.

<|endoftext|>

Once upon a time, in a warm and sunny place, there was a big pit. A little boy named Tom liked to play near the pit. One day, Tom lost his red ball. He was very sad.

Tom asked his friend, Sam, to help him search for the ball. They looked high and low, but they could not find the ball. Tom said, "I think my ball fell into the pit."

Sam and Tom went close to the pit. They were scared, but they wanted to find the red ball. They looked into the pit, but it was too dark to see. Tom said, "We must go in and search for my ball."



In [8]:
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

import regex as re
re.findall(PAT, "Some text that i'll pre-tokenize")

['Some', ' text', ' that', ' i', "'ll", ' pre', '-', 'tokenize']

## Training a Tokenizer (small sample)

In [ ]:
# Train a byte-level BPE tokenizer on the TinyStories dataset
from bpe_slow.train_bpe_parallel import train_bpe_parallel as _train_bpe
import json

PATH_TS = "data/TinyStoriesV2-GPT4-valid.txt"
vocab, merges = _train_bpe(PATH_TS, vocab_size=10000, special_tokens=["<|endoftext|>"])

with open("outputs/vocab_tinystories.json", "w", encoding="utf-8") as f:
    json.dump({str(k): v.decode("latin1") for k, v in vocab.items()}, f, indent=2)

# 保存 merges.json
with open("outputs/merges_tinystories.json", "w", encoding="utf-8") as f:
    json.dump([[a.decode("latin1"), b.decode("latin1")] for a, b in merges], f, indent=2)

In [9]:
with open("vocab.json", "w", encoding="utf-8") as f:
    json.dump({str(k): v.decode("latin1") for k, v in vocab.items()}, f, indent=2)

# 查看最长的 token
longest_token = max(vocab.values(), key=len)
print(f"Longest token: {longest_token} with length {len(longest_token)}")

Longest token: b' accomplishment' with length 15


## Train a BPE tokenizer on TinyStories dataset
- Compare the performances

In [ ]:
# Train a slow byte-level BPE tokenizer on the TinyStories dataset - requires 7min
from bpe_slow.train_bpe_parallel import train_bpe_parallel as _train_bpe_parallel
import json

PATH_TS = "data/TinyStoriesV2-GPT4-train.txt"
vocab, merges = _train_bpe_parallel(PATH_TS, vocab_size=10000, special_tokens=["<|endoftext|>"], num_workers=4)

with open("outputs/vocab_tinystories.json", "w", encoding="utf-8") as f:
    json.dump({str(k): v.decode("latin1") for k, v in vocab.items()}, f, indent=2)

# 保存 merges.json
with open("outputs/merges_tinystories.json", "w", encoding="utf-8") as f:
    json.dump([[a.decode("latin1"), b.decode("latin1")] for a, b in merges], f, indent=2)

100%|██████████| 9743/9743 [07:31<00:00, 21.60it/s]


In [ ]:
# Train a byte-level BPE tokenizer on the TinyStories dataset - requires less than 1s
from bpe_optimized.train_bpe_parallel_optimized import train_bpe_parallel
import json

PATH_TS = "data/TinyStoriesV2-GPT4-train.txt"
vocab, merges = train_bpe_parallel(PATH_TS, vocab_size=10000, special_tokens=["<|endoftext|>"], num_workers=4)
with open("outputs/vocab_tinystories_optimized.json", "w", encoding="utf-8") as f:
    json.dump({str(k): v.decode("latin1") for k, v in vocab.items()}, f, indent=2)

# 保存 merges.json
with open("outputs/merges_tinystories_optimized.json", "w", encoding="utf-8") as f:
    json.dump([[a.decode("latin1"), b.decode("latin1")] for a, b in merges], f, indent=2)

100%|██████████| 9743/9743 [00:00<00:00, 26789.96it/s]


- Efficiency increase: from 21.60it/s to **26789.96it/s**

## Train a BPE Tokenizer on OpenWebText dataset (1GB subset)

In [ ]:
# Train a byte-level BPE tokenizer on the owt dataset
size_subset = 1024
PATH_OWT = "data/owt_train.txt"
PATH_OWT_SUBSET = f"data/owt_subset_{size_subset}mb.txt"

with open(PATH_OWT, "rb") as f_in:
    chunk = f_in.read(size_subset * 1024 * 1024) # only read 1024 mb
    with open(PATH_OWT_SUBSET, "wb") as f_out:
        f_out.write(chunk)


from bpe_optimized.train_bpe_parallel_optimized import train_bpe_parallel
#from train_bpe import train_bpe  - too slow, need 19 hrs to train
import json

PATH_OWT = "data/owt_train.txt"
vocab, merges = train_bpe_parallel(
    PATH_OWT_SUBSET, 
    vocab_size=32000, 
    special_tokens=["<|endoftext|>"], 
    num_workers=4
)

with open("outputs/vocab_owt.json", "w", encoding="utf-8") as f:
    json.dump({str(k): v.decode("latin1") for k, v in vocab.items()}, f, indent=2)

# save merges.json
with open("outputs/merges_owt.json", "w", encoding="utf-8") as f:
    json.dump([[a.decode("latin1"), b.decode("latin1")] for a, b in merges], f, indent=2)

已经提取大小为 1024 mb 的 owt 子集


100%|██████████| 31743/31743 [00:07<00:00, 4429.60it/s] 


- The optimized version reduces the 19 hrs training time to 7 seconds

## Using the Tokenizer: Sample 10 documents from datasets and tokenize them (OWT)

In [ ]:
import numpy as np
import json
with open("data/owt_valid.txt", "r") as f:
    lines = f.readlines()
    samples = lines[:10]

In [ ]:
from bpe_optimized.tokenizer_optimized import Tokenizer
# from bpe_slow.tokenizer import Tokenizer

special_tokens = ["<|endoftext|>"]
tokenizer = Tokenizer.from_files("outputs/vocab_owt.json", "outputs/merges_owt.json", special_tokens)

for sample in samples:
    encoded_tokens = tokenizer.encode(sample)
    print("="*80)
    print(sample)
    print(encoded_tokens)

LOUISVILLE, Ky. — A few unflattering reviews are to be expected with any hotel, particularly one whose rates start at $49 per night. But while complaints about shabby rooms and thin towels are common in the industry, ones like these, from TripAdvisor.com, are not: “It is a clean hotel but there are a lot of homeless people there.” “Run far far away!!!!! This is a homeless shelter, not a hotel!” “DO NOT STAY HERE UNLESS YOU ARE HOMELESS… All of the workers are former addicts/homeless people.” Hotel Louisville, 12 stories of brick adorned with a large white cross, is indeed a hotel and event space open to the public. At the same time, it is a transitional-housing facility, substance-abuse recovery center and job-training site owned and operated by Wayside Christian Mission, a nonprofit that shelters and feeds the city’s homeless population. Wayside bought the building at a foreclosure auction in 2009, never intending to rent rooms to the general public. It was simply a place to house the

In [ ]:
# Estimate the throughput of the tokenizer ((e.g., in bytes/second))

import time
start = time.time()
for sample in samples:
    encoded_tokens = tokenizer.encode(sample)
end = time.time()
total_bytes = sum(len(s.encode("utf-8")) for s in samples)
throughput = total_bytes / (end-start)
throughput 

# naive version:        830 bytes per sec
# optimized version:    2526412 bytes per sec.

2526412.8